# Data Analysis & Modeling: Predict Next-Week Play Time

**Original interview mock, not a reported Roblox question or an official CodeSignal assessment.**
Interview format: **5 minutes introduction + 35 minutes coding + 5 minutes Q&A**. All data are synthetic.

## The interviewer prompt

You work on a game platform. At the start of day 0, the team wants to estimate how many minutes each player will spend playing during the next seven days. You receive a player table and a session log.

Use only information available before day 0. Predict **next_week_minutes**, the total minutes during days 0–6. For example, a prediction of 75 means an estimated 75 minutes over that week. A player who does not play has a valid target of zero.

**Problem 1 — Data analysis (14 minutes).** Load both CSVs, inspect data quality, clean the session log, and build one row per player. Report the fraction of players without historical sessions and average observed historical play time by device.

**Problem 2 — Modeling (21 minutes).** Select features, create a train/test split, fit a simple regression model and a constant baseline, evaluate their errors, and predict for a new player. Explain your choices as you code. No neural network or from-scratch optimizer is required.

| Coding time | Task |
| --- | --- |
| 0–4 min | Load and inspect |
| 4–9 min | Clean the data |
| 9–14 min | Analyze and prepare features |
| 14–18 min | Select features and split |
| 18–27 min | Preprocess and train |
| 27–32 min | Evaluate |
| 32–35 min | Predict and summarize |

These are suggested checkpoints, not official sub-round timings. Ask during the introduction: What is one example? When is a prediction made? What exactly is the target?

**Reference version.** Run cells top to bottom. Code is complete; each step includes checks and expandable follow-up answers. Try the matching practice notebook first.

## Data contract — supplied by the interviewer

**players.csv:** one row per player. Player attributes are snapshots taken before day 0. The export was assembled after every player's outcome window finished, so the target is known for training. Keep every player, including those without historical sessions.

| Column | Meaning |
| --- | --- |
| user_id | Unique player identifier; used for joins, not as a numeric predictor |
| account_age_days | Account age at the prediction cutoff; may be missing |
| country | Country category; may be missing |
| device | Device category; may be missing |
| next_week_minutes | Target: total minutes played during days 0–6; zero is valid |
| returned_next_week | Whether the player returns during days 0–6; a future outcome, not an input feature |

**sessions.csv:** one row per logged session before cleaning. A player can have multiple sessions or none.

| Column | Meaning |
| --- | --- |
| session_id | Session identifier; the export may repeat identical rows |
| user_id | Player identifier; always refers to a player in players.csv |
| day_offset | Day relative to the cutoff: -1 means yesterday, 0 means the prediction day; only -7 through -1 are valid history |
| minutes_played | Session duration; missing or a negative sentinel means unknown, not zero |
| completed | 1 if the session was completed, otherwise 0; remains valid when duration is unknown |

**Aggregation rules for this exercise:**

- Keep sessions whose durations are unknown: they still count as sessions.
- With no historical sessions, set session_count, known_duration_count and total_minutes to 0; leave avg_minutes and completion_rate missing.
- With sessions but no known durations, leave total_minutes and avg_minutes missing.
- With some known durations, total_minutes sums the observed durations only; it can understate true play time.

Distinct players at this shared cutoff are assumed to be independent samples for the mock. The random holdout estimates performance on other players from this population; it does not measure robustness to changes over future weeks.


## Provided setup

Open this notebook from the workspace root in VS Code or Jupyter and select **.venv-codesignal/bin/python**. No network, GPU or additional package installation is required.

The two input files are **data/codesignal_roblox/players.csv** and **data/codesignal_roblox/sessions.csv**. The provided cell imports libraries and sets DATA_DIR, a pathlib.Path directory object. In Step 1, use pd.read_csv(DATA_DIR / "players.csv") and pd.read_csv(DATA_DIR / "sessions.csv") to load them. Do not modify the source CSVs.

Start the 35-minute coding timer after setup.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import (mean_absolute_error, root_mean_squared_error, r2_score,
    accuracy_score, f1_score, log_loss, confusion_matrix, classification_report,
    roc_auc_score, average_precision_score, precision_score, recall_score)
SEED = 42

DATA_DIR = Path("data/codesignal_roblox")
assert (DATA_DIR / "players.csv").exists(), "Run with the workspace root as your working directory."


## Problem 1 · Step 1 — Load and inspect (0–4 min)

Load **players** and **sessions** with pd.read_csv. Inspect their shapes, sample rows, column types and missing values. Check unique player IDs, exact duplicate sessions, negative durations and records outside days -7 through -1.

Inspect **next_week_minutes** with describe, including the 50th, 90th and 99th percentiles. Report the fraction of labels equal to zero. A high upper percentile can reveal a long tail: a small number of players with much larger values.

**Say before coding:** "I will first check what each row represents, which fields are available at prediction time, and whether the target contains missing or invalid values."

**Possible interviewer questions**

1. What does one training example represent, and what is its target?
2. Can returned_next_week be a feature?
3. What should you inspect before dropping missing rows?


In [2]:
players = pd.read_csv(DATA_DIR / "players.csv")
sessions = pd.read_csv(DATA_DIR / "sessions.csv")
players.info()
display(players.head(), sessions.head(), players.isna().sum(), sessions.isna().sum())
display(players["next_week_minutes"].describe(percentiles=[0.5, 0.9, 0.99]))
print("zero target fraction:", players["next_week_minutes"].eq(0).mean())
print("exact duplicate sessions:", sessions.duplicated().sum())
print("outside history:", (~sessions["day_offset"].between(-7, -1)).sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   user_id             1000 non-null   object 
 1   account_age_days    970 non-null    float64
 2   country             950 non-null    object 
 3   device              950 non-null    object 
 4   returned_next_week  1000 non-null   int64  
 5   next_week_minutes   1000 non-null   float64
dtypes: float64(2), int64(1), object(3)
memory usage: 47.0+ KB


,user_id,account_age_days,country,device,returned_next_week,next_week_minutes
0,user_0001,184.0,US,mobile,0,0.00
1,user_0002,6.0,US,mobile,0,0.00
2,user_0003,56.0,US,desktop,0,0.00
3,user_0004,39.0,US,desktop,1,247.43
4,user_0005,295.0,GB,NaN,1,5.96


,session_id,user_id,day_offset,minutes_played,completed
0,session_01326,user_0274,-7,84.25,1
1,session_03494,user_0706,-7,130.05,1
2,session_04282,user_0872,-1,33.65,1
3,session_04588,user_0937,-3,74.44,1
4,session_00441,user_0092,-1,66.84,1


user_id                0
account_age_days      30
country               50
device                50
returned_next_week     0
next_week_minutes      0
dtype: int64

session_id         0
user_id            0
day_offset         0
minutes_played    99
completed          0
dtype: int64

count    1000.000000
mean       43.316520
std        82.095041
min         0.000000
50%         0.000000
90%       192.697000
99%       290.693100
max       299.290000
Name: next_week_minutes, dtype: float64

zero target fraction: 0.707
exact duplicate sessions: 30
outside history: 30


In [3]:
assert players["user_id"].is_unique
assert players["next_week_minutes"].notna().all()
assert players["next_week_minutes"].ge(0).all()
assert sessions["user_id"].isin(players["user_id"]).all()


<details>
<summary>Follow-up answers</summary>

**What does one training example represent, and what is its target?**

One player at the start of day 0. The target is that player's total minutes played during days 0–6. Session rows must be aggregated to player level before training.

**Can returned_next_week be a feature?**

No. It is an outcome from the future prediction window. The offline export contains it, but the serving system would not know it at day 0.

**What should you inspect before dropping missing rows?**

Check which columns are missing and what missingness means. Missing predictors do not automatically invalidate the label or the other fields.

</details>


## Problem 1 · Step 2 — Clean the data (4–9 min)

Create **clean_sessions** without modifying **sessions**. Remove exact duplicate rows, retain day_offset values from -7 through -1 inclusive, and replace negative minutes_played with NaN. Keep rows with unknown durations.

After removing exact duplicates, check that session_id is unique. Conflicting records with the same ID require investigation rather than silently keeping an arbitrary row. Do not fill missing values with learned statistics yet.

**Say before coding:** "I am applying fixed validity rules, not learning statistics from the full dataset."

**Possible interviewer questions**

1. Why keep a session whose duration is missing?
2. Why not replace missing durations with zero?
3. Why postpone median imputation until after splitting?


In [4]:
clean_sessions = sessions.drop_duplicates().copy()
clean_sessions = clean_sessions.loc[clean_sessions["day_offset"].between(-7, -1)].copy()
assert clean_sessions["session_id"].is_unique
clean_sessions["minutes_played"] = clean_sessions["minutes_played"].mask(
    clean_sessions["minutes_played"].lt(0)
)


In [5]:
assert clean_sessions["day_offset"].between(-7, -1).all()
assert clean_sessions["minutes_played"].dropna().ge(0).all()
assert clean_sessions["session_id"].is_unique


<details>
<summary>Follow-up answers</summary>

**Why keep a session whose duration is missing?**

The session still happened and its completion indicator can still be useful.

**Why not replace missing durations with zero?**

Unknown duration is not evidence that a session lasted zero minutes.

**Why postpone median imputation until after splitting?**

A median is learned from data. Fit it on training rows only; the held-out rows must not influence preprocessing.

</details>


## Problem 1 · Step 3 — Analyze and prepare features (9–14 min)

Group **clean_sessions** by user_id and create **activity** with:

- **session_count:** number of sessions, including unknown durations.
- **known_duration_count:** number of nonmissing durations.
- **total_minutes:** sum of observed durations; missing if every duration is unknown.
- **avg_minutes:** mean of observed durations.
- **completion_rate:** mean of the 0/1 completed indicator.

Left join activity into players to create **features**, one row per player. Create **no_history**, a Boolean mask for players without matching sessions, and apply the aggregation rules in the data contract.

Report the no-history fraction. Group features by device and report player count, count of observed totals, and mean observed total_minutes. Include players without history and retain a missing-device group.

**Say before coding:** "I need one training row per player, including people without any history."

**Possible interviewer questions**

1. Why a left join?
2. Can a partial total understate activity?
3. How do you check that a merge did not multiply rows?


In [6]:
activity = clean_sessions.groupby("user_id").agg(
    session_count=("session_id", "size"),
    known_duration_count=("minutes_played", "count"),
    total_minutes=("minutes_played", "sum"),
    avg_minutes=("minutes_played", "mean"),
    completion_rate=("completed", "mean"),
)
activity.loc[activity["known_duration_count"].eq(0), "total_minutes"] = np.nan
features = players.merge(activity, on="user_id", how="left", validate="one_to_one")
no_history = features["session_count"].isna()
features.loc[no_history, ["session_count", "known_duration_count", "total_minutes"]] = 0
display(features.groupby("device", dropna=False).agg(
    players=("user_id", "size"),
    observed_totals=("total_minutes", "count"),
    mean_observed_minutes=("total_minutes", "mean"),
))
print("no-history fraction:", no_history.mean())


,players,observed_totals,mean_observed_minutes
device,,,
console,79,79,174.474937
desktop,279,278,172.055000
mobile,442,441,168.966735
tablet,150,149,167.552416
NaN,50,50,163.201600


no-history fraction: 0.093


In [7]:
assert len(features) == len(players)
assert features["user_id"].is_unique
assert features.loc[no_history, "total_minutes"].eq(0).all()
unknown = features["session_count"].gt(0) & features["known_duration_count"].eq(0)
assert features.loc[unknown, "total_minutes"].isna().all()


<details>
<summary>Follow-up answers</summary>

**Why a left join?**

Keep every labeled player, including those with no matching historical session.

**Can a partial total understate activity?**

Yes. Keep known_duration_count so missingness is visible; do not claim the sum is the true total.

**How do you check that a merge did not multiply rows?**

Declare the expected key relationship with validate, then check row counts and the training-example identifier.

</details>


## Problem 2 · Step 4 — Select features and split (14–18 min)

Define **numeric_features** as account_age_days, session_count, known_duration_count, total_minutes, avg_minutes and completion_rate. Define **categorical_features** as country and device.

Create **X** using only these columns and **y** from next_week_minutes. Exclude user_id and both future-outcome columns from X. Create **X_train**, **X_test**, **y_train**, **y_test** with an 80/20 train_test_split and random_state=SEED. Do not stratify directly on the continuous target.

**Say before coding:** "I will keep the holdout untouched and learn preprocessing only on training players."

**Possible interviewer questions**

1. Should we stratify by this continuous target?
2. What if rows contain repeated cutoffs for the same user?
3. How do you check that X and y still correspond?


In [8]:
numeric_features = ["account_age_days", "session_count", "known_duration_count",
                    "total_minutes", "avg_minutes", "completion_rate"]
categorical_features = ["country", "device"]
X = features[numeric_features + categorical_features]
y = features["next_week_minutes"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)


In [9]:
assert set(X_train.index).isdisjoint(X_test.index)
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert not {"user_id", "returned_next_week", "next_week_minutes"} & set(X.columns)


<details>
<summary>Follow-up answers</summary>

**Should we stratify by this continuous target?**

Not directly: many target values are unique. A simple random split is sufficient under this mock contract.

**What if rows contain repeated cutoffs for the same user?**

Choose a split matching deployment, often chronological with non-overlapping outcome windows; consider user grouping for a new-user evaluation.

**How do you check that X and y still correspond?**

Select both from the same ordered table and apply the same split indices. Verify their indices match within each subset.

</details>


## Problem 2 · Step 5 — Preprocess and train (18–27 min)

Fit **baseline**, a DummyRegressor(strategy="mean"), which predicts the training-label mean for every player.

Create **numeric_pipeline** with SimpleImputer(strategy="median") followed by StandardScaler. Create **categorical_pipeline** with SimpleImputer(strategy="most_frequent") followed by OneHotEncoder(handle_unknown="ignore"). Combine them in a ColumnTransformer named **preprocess**.

Create **model**, a Pipeline with preprocess and a regressor named **regressor**, using Ridge(alpha=1.0). Fit only X_train and y_train. Ridge predicts a weighted sum of the transformed features plus an intercept; its L2 penalty discourages large coefficients.

**Say before coding:** "I will compare a constant training-mean baseline with a regularized linear model. The pipeline will learn imputation, scaling and encoding from training data only."

**Possible interviewer questions**

1. Why scale for Ridge?
2. Is the mean the best constant for every metric?
3. What happens when you call model.fit on the Pipeline?


In [10]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])
model = Pipeline([("preprocess", preprocess), ("regressor", Ridge(alpha=1.0))])
model.fit(X_train, y_train)
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)


In [11]:
assert model.named_steps["regressor"].coef_.size > 0
learned = model.named_steps["preprocess"].named_transformers_["num"].named_steps["imputer"].statistics_
np.testing.assert_allclose(learned, X_train[numeric_features].median().to_numpy())


<details>
<summary>Follow-up answers</summary>

**Why scale for Ridge?**

The penalty acts on coefficients. Comparable feature scales make regularization more comparable across features.

**Is the mean the best constant for every metric?**

The mean minimizes training squared error; the median minimizes training absolute error. A median baseline is an optional MAE comparison.

**What happens when you call model.fit on the Pipeline?**

The preprocessing steps fit and transform the training features. The final estimator then fits those transformed features against the training labels.

</details>


## Problem 2 · Step 6 — Evaluate (27–32 min)

Create **pred** and **baseline_pred** by predicting on X_test. The application requires nonnegative minutes, so apply np.maximum(prediction, 0) to both outputs as a fixed serving rule, not a choice tuned on test results.

Create a **scores** table with rows **mean_baseline** and **ridge** and columns **MAE_minutes**, **RMSE_minutes** and **R2**:

- **MAE:** average absolute prediction error, in minutes; lower is better.
- **RMSE:** square root of average squared error, in minutes; gives more weight to large errors; lower is better.
- **R2:** 1 minus model squared error divided by squared error around the test-label mean; higher is better. It can be negative.

Create **errors** containing actual_minutes, predicted_minutes and absolute_error. Display the five largest errors and give a short interpretation. Do not change hyperparameters based on this test set.

**Say before coding:** "I will report errors in minutes, compare against the constant baseline, and inspect where the model misses badly. I will keep model selection separate from final test evaluation."

**Possible interviewer questions**

1. Can R2 be negative?
2. What if the target has many zeros and a long tail?
3. Can you keep changing the model after looking at test metrics?


In [12]:
pred = np.maximum(model.predict(X_test), 0)
baseline_pred = np.maximum(baseline.predict(X_test), 0)
def evaluate_regression(actual, prediction):
    return {"MAE_minutes": mean_absolute_error(actual, prediction),
            "RMSE_minutes": root_mean_squared_error(actual, prediction),
            "R2": r2_score(actual, prediction)}
scores = pd.DataFrame({"mean_baseline": evaluate_regression(y_test, baseline_pred),
                       "ridge": evaluate_regression(y_test, pred)}).T
display(scores)
errors = pd.DataFrame({"actual_minutes": y_test, "predicted_minutes": pred})
errors["absolute_error"] = (errors["actual_minutes"] - errors["predicted_minutes"]).abs()
display(errors.nlargest(5, "absolute_error"))


,MAE_minutes,RMSE_minutes,R2
mean_baseline,63.426487,81.777893,-0.000123
ridge,57.482557,77.936300,0.091634


,actual_minutes,predicted_minutes,absolute_error
462,288.60,19.917828,268.682172
78,290.69,59.217425,231.472575
737,241.78,22.779150,219.000850
309,250.64,31.657494,218.982506
136,272.78,59.358661,213.421339


In [13]:
assert pred.shape == (len(y_test),)
assert np.isfinite(pred).all() and (pred >= 0).all()
np.testing.assert_allclose(scores.loc["ridge", "MAE_minutes"], np.abs(y_test.to_numpy() - pred).mean())
np.testing.assert_allclose(scores.loc["ridge", "RMSE_minutes"], np.sqrt(np.mean((y_test.to_numpy() - pred) ** 2)))


<details>
<summary>Follow-up answers</summary>

**Can R2 be negative?**

Yes. It means squared error is worse than predicting the test-set mean. This definition is distinct from comparing against our training-mean deployable baseline.

**What if the target has many zeros and a long tail?**

Discuss a log1p target or a two-stage return-probability times conditional-duration model. Select alternatives on validation data, not this test set.

**Can you keep changing the model after looking at test metrics?**

Use a validation split or cross-validation for those choices. Repeatedly selecting against the test set makes its reported performance optimistic.

</details>


## Problem 2 · Step 7 — Predict and summarize (32–35 min)

Create **new_player**, a one-row DataFrame with these already-aggregated pre-cutoff features:

| Feature | Value |
| --- | --- |
| account_age_days | np.nan |
| session_count | 3 |
| known_duration_count | 2 |
| total_minutes | 55 |
| avg_minutes | 27.5 |
| completion_rate | 2/3 |
| country | "UNSEEN_COUNTRY" |
| device | "mobile" |

Use the fitted model to create **new_prediction**, clip it at zero using the serving rule, and report the result in minutes. Do not fit again. Give a 30-second summary of the input features, split, model, evaluation, one limitation and one next experiment.

**Say before coding:** "Inference reuses the training transformations. An unseen category must not crash the pipeline."

**Possible interviewer questions**

1. Why not call fit_transform on the new player?
2. Does better predicted play time imply a better product?
3. What must training and inference share?


In [14]:
new_player = pd.DataFrame([{
    "account_age_days": np.nan, "session_count": 3, "known_duration_count": 2,
    "total_minutes": 55, "avg_minutes": 27.5, "completion_rate": 2/3,
    "country": "UNSEEN_COUNTRY", "device": "mobile",
}])
new_prediction = np.maximum(model.predict(new_player), 0)
print("Predicted next-week minutes:", new_prediction[0])


Predicted next-week minutes: 26.254611542309085


In [15]:
assert new_prediction.shape == (1,)
assert np.isfinite(new_prediction).all()


<details>
<summary>Follow-up answers</summary>

**Why not call fit_transform on the new player?**

That would replace training statistics and category definitions. Use the fitted pipeline predict method.

**Does better predicted play time imply a better product?**

No. Engagement is a proxy; quality, safety and user satisfaction need separate guardrails and online evaluation.

**What must training and inference share?**

The feature definitions, column meanings and fitted preprocessing. Reuse the saved pipeline rather than relearning imputation, scaling or categories.

</details>


## If the interviewer extends the problem

These questions are optional, outside the 35-minute core.

1. **MAE is the business metric. Which constant baseline would you choose?** A training-label median minimizes training absolute error. Compare it with the training-mean baseline, which minimizes squared error.
2. **A few players have extremely large targets. What would you try?** Check that the values are valid, then consider a log1p target or a different model on validation data. Evaluate final predictions in minutes; exponentiating a predicted mean log value does not generally recover the mean in the original units.
3. **Does the model work equally well across devices?** Report per-device errors together with group counts. Small groups have uncertain estimates; avoid concluding that a device causes an error difference.
4. **How would you improve the model?** Try one tree-based model on a validation split, compare it with Ridge, and keep the final test set untouched until the choice is fixed.
5. **How would you evaluate future weeks?** Collect multiple prediction cutoffs, ensure every feature precedes its cutoff, and hold out a later period with fully observed outcome windows.


## References and provenance

This is an original synthetic regression exercise, not a reported Roblox question or an official CodeSignal assessment. The interview format follows the user-supplied **Data Analysis & Modeling Coding Interview** guide: 5 minutes introduction, 35 minutes for data analysis and model training, and 5 minutes Q&A. Step timings are suggested practice checkpoints.

The task, synthetic data and reference implementation are practice material. They do not describe actual Roblox player behavior or guarantee that this topic will be asked.

Implementation references: [scikit-learn model evaluation](https://scikit-learn.org/1.6/modules/model_evaluation.html), [avoiding data leakage](https://scikit-learn.org/1.6/common_pitfalls.html#data-leakage).
